# Malware Dataset Overview

This notebook summarizes the malware dataset, including total number of flows, number of malware families, and overall temporal coverage. It provides a structural understanding of dataset scope and composition.

In [1]:
%reload_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import sys
sys.path.append('../../src')
from tls_profiling.io import readers 

## Data Loading and Validation

The dataset is loaded and inspected for schema consistency, missing values, and correct parsing of metadata fields (e.g., malware family labels). This ensures reliability of subsequent analysis.

In [2]:
print("Opening dataset...")
dataset = readers.open_parquet_dataset(f"../../datasets/malware.parquet")

print(f"Loading dataset (first 100k rows)...")
table_100k = dataset.head(100_000)
df_100k = table_100k.to_pandas()
df_10k = df_100k.sample(n=10_000, random_state=42)
df_1k = df_100k.sample(n=1_000, random_state=42)
print(f"Data loaded!")
df=df_100k  # set 100k dataset as working dataset

Opening dataset...
Loading dataset (first 100k rows)...
Data loaded!


Use existing information to refine the connection label.

In [4]:
from tls_profiling.exploration.connections import get_connection_label, remove_grease_values

# Label connections using process information
system_processes = {"System", "svchost.exe", "msedge.exe", "backgroundTaskHost.exe", "Explorer.EXE", "explorer.exe", "smartscreen.exe"}
unknown_processes = {"", None}

df["connection_label"] = get_connection_label(df, "by_malware_family", None)
df = remove_grease_values(df)

## Malware Family Distribution

Compute the number of flows per malware family. This reveals class imbalance, dominant families, and rare samples within the dataset.

In [5]:
from tls_profiling.exploration.connections import get_connections_per_x
connection_per_malware = get_connections_per_x(df, "meta.malware.family")
connection_per_malware

,meta.malware.family,connections,system_connections,unknown_connections,application_connections,malware_connections,processes
0,agenttesla,70,0,0,0,70,[]
1,ardamax,6,0,0,0,6,[]
2,asyncrat,66,0,0,0,66,[]
3,azorult,16,0,0,0,16,[]
4,betabot,5,0,0,0,5,[]
5,blackmoon,28,0,0,0,28,[]
6,cybergate,29,0,0,0,29,[]
7,darkcomet,20,0,0,0,20,[]
8,formbook,75,0,0,0,75,[]
9,guloader,12,0,0,0,12,[]


## Temporal Distribution of Malware Activity

Analyze how malicious connections are distributed over time. This helps identify bursts of activity, campaign-like behavior, or irregular collection patterns.

In [6]:
from tls_profiling.exploration.connections import get_weekly_connections_per_x
weekly_per_applications = get_weekly_connections_per_x(df, "meta.malware.family")
weekly_per_applications

connection_label,meta.malware.family,week,malware,connections
0,agenttesla,2024-01-01,70,0
1,ardamax,2024-01-01,6,0
2,asyncrat,2024-01-01,66,0
3,azorult,2024-01-01,16,0
4,betabot,2024-01-01,5,0
5,blackmoon,2024-01-01,28,0
6,cybergate,2024-01-01,29,0
7,darkcomet,2024-01-01,20,0
8,formbook,2024-01-01,75,0
9,guloader,2024-01-01,12,0


## Categorical Field Cardinality

Compute the number of unique values for categorical TLS attributes (e.g., SNI, TLS version, cipher suite, JA3). This quantifies variability within malicious traffic.

In [7]:
import tls_profiling.exploration.cardinality as c

cardinality_df = c.get_cardinality_stat(df)
cardinality_df

,field,cardinality,total_rows,top1_value,top1_freq,top1_ratio,top2_value,top2_freq,top2_ratio,top3_value,top3_freq,top3_ratio
0,da,441,100000,204.79.197.200,11996.0,0.1200,20.123.104.105,6394.0,0.0639,20.223.35.26,4135.0,0.0414
1,sa,246,100000,10.127.0.3,1946.0,0.0195,10.127.0.32,1640.0,0.0164,10.127.0.16,1633.0,0.0163
2,tls.cver,4,100000,0303,96527.0,0.9653,0301,2584.0,0.0258,0300,870.0,0.0087
3,tls.sver,3,100000,0303,87208.0,0.8721,0301,477.0,0.0048,0302,18.0,0.0002
4,tls.scs,13,100000,C030,60354.0,0.6035,C02C,19265.0,0.1926,C02F,6277.0,0.0628
5,tls.sni,178,100000,login.live.com,17616.0,0.1762,arc.msn.com,13009.0,0.1301,settings-win.data.microsoft.com,12907.0,0.1291
6,tls.ja3,22,100000,28a2c9bd18a11de089ef85a160da29e4,88893.0,0.8889,a0e9f5d64349fb13191bc781f81f42e1,6301.0,0.0630,1d095e68489d3c535297cd8dffb06cb9,1285.0,0.0128
7,tls.ja4,24,100000,t12d1909h2_d83cc789557e_7af1ed941c26,88893.0,0.8889,t12d190800_d83cc789557e_7af1ed941c26,6300.0,0.0630,t10d120400_d94e65cdb899_f8ec56bc740a,1285.0,0.0128
8,tls.ja3s,99,100000,87b9bed0515be16c82bdc44d88a4a0ec,16909.0,0.1691,7d8fd34fdb13a7fff30d5a52846b6c4c,16255.0,0.1626,67bfe5d15ae567fb35fd7837f0116eec,13713.0,0.1371
9,tls.ja4s,92,100000,t1204h2_c02c_5333cdffa7d9,16909.0,0.1691,t120400_c030_09f674154ab3,16255.0,0.1626,t1204h2_c030_09f674154ab3,13713.0,0.1371


## Cardinality per Malware Family

Measure attribute diversity per family. Low diversity may indicate stable tooling; high diversity may suggest polymorphism or infrastructure variation.

In [9]:
import tls_profiling.exploration.cardinality as c

cardinality_per_malware = c.get_cardinality_stat_per_x(df, "meta.malware.family")
display("Malware connections")
display(cardinality_per_malware)

'Malware connections'

,tls.cver,tls.sver,tls.sext,tls.csg,tls.ccs,tls.cext,tls.ssv,tls.csv,tls.scs,tls.alpn,tls.sni,tls.ja3,tls.ja4,tls.ja3s,tls.ja4s,tls.rec,connections
meta.malware.family,,,,,,,,,,,,,,,,,
agenttesla,1,1,4,2,3,3,1,1,2,2,4,3,3,4,3,30,70
ardamax,1,1,1,1,1,1,1,1,1,1,2,1,1,1,1,4,6
asyncrat,2,2,6,3,4,6,1,1,4,2,7,6,6,9,9,28,66
azorult,1,1,2,1,1,1,1,1,1,1,2,1,1,2,1,9,16
betabot,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,5
blackmoon,1,1,2,1,1,1,1,1,1,1,2,1,1,2,1,15,28
cybergate,1,1,2,1,1,1,1,1,1,1,2,1,1,2,1,13,29
darkcomet,1,1,1,1,1,1,1,1,1,1,2,1,1,1,1,9,20
formbook,1,1,1,1,1,1,1,1,1,1,2,1,1,1,1,32,75


## TLS Protocol Characteristics

Summarize distributions of TLS versions, cipher suites, and extensions used by malware. This characterizes cryptographic behavior and protocol preferences of malicious traffic.

In [10]:
from tls_profiling.exploration.cardinality import get_df_tls_field_card
display(get_df_tls_field_card(df, "tls.cver").sort_values("count", ascending=False))
display(get_df_tls_field_card(df, "tls.sver").sort_values("count", ascending=False))
display(get_df_tls_field_card(df, "tls.alpn").sort_values("count", ascending=False))
display(get_df_tls_field_card(df, "tls.csg").sort_values("count", ascending=False))
display(get_df_tls_field_card(df, "tls.ccs").sort_values("count", ascending=False))
display(get_df_tls_field_card(df, "tls.cext").sort_values("count", ascending=False))
display(get_df_tls_field_card(df, "tls.ja4").sort_values("count", ascending=False))

,tls.cver,count,ratio
3,0303,96527,0.96527
1,0301,2584,0.02584
0,0300,870,0.00870
2,0302,19,0.00019


,tls.sver,count,ratio
2,0303,87208,0.87208
0,0301,477,0.00477
1,0302,18,0.00018


,tls.alpn,count,ratio
1,"h2,http/1.1",89192,0.89192
0,,10808,0.10808


,tls.csg,count,ratio
7,"0804,0805,0806,0401,0501,0201,0403,0503,0203,0...",95436,0.95436
0,,3473,0.03473
2,"0401,0501,0201,0403,0503,0203,0202",774,0.00774
6,"0804,0403,0807,0805,0806,0401,0501,0601,0503,0...",168,0.00168
5,"0403,0804,0401,0503,0805,0501,0806,0601",131,0.00131
4,"0403,0503,0603,0807,0808,0809,080A,080B,0804,0...",13,0.00013
1,"0401,0403,0501,0503,0601,0603,0301,0303,0201,0203",3,0.00003
3,"0401,0501,0201,0403,0503,0203,0202,0601,0603",2,0.00002


,tls.ccs,count,ratio
7,"C02C,C02B,C030,C02F,C024,C023,C028,C027,C00A,C...",95206,0.95206
1,"002F,0035,0005,000A,C013,C014,C009,C00A,0032,0...",1451,0.01451
5,"C00A,C009,C014,C013,0035,002F,000A",1152,0.01152
0,"0005,000A,0013,0004,00FF",870,0.00870
2,"003C,002F,003D,0035,0005,000A,C027,C013,C014,C...",774,0.00774
6,"C02C,C02B,C030,C02F,009F,009E,C024,C023,C028,C...",232,0.00232
9,"C02F,C030,C02B,C02C,CCA8,CCA9,C013,C009,C014,C...",168,0.00168
3,"1301,1302,1303,C02B,C02F,C02C,C030,CCA9,CCA8,C...",131,0.00131
4,"1302,1303,1301,C02C,C030,009F,CCA9,CCA8,CCAA,C...",13,0.00013
8,"C02C,C087,CCA9,C0AD,C00A,C02B,C086,C0AC,C009,C...",3,0.00003


,tls.cext,count,ratio
1,"0000,0005,000A,000B,000D,0023,0010,0017,FF01",88893,0.88893
2,"0000,0005,000A,000B,000D,0023,0017,FF01",6304,0.06304
17,"FF01,0000,000A,000B",1304,0.01304
3,"0000,0005,000A,000B,0023,0017,FF01",1077,0.01077
0,,870,0.00870
16,"FF01,0000,0005,000A,000B,000D",696,0.00696
4,"0000,000A,000B,000D,0023,0017,FF01",229,0.00229
14,"3374,0000,0005,000A,000B,000D,FF01,0010,0012,0...",168,0.00168
19,"FF01,000A,000B",127,0.00127
7,"0000,0017,FF01,000A,000B,0023,0010,0005,000D,0...",118,0.00118


,tls.ja4,count,ratio
8,t12d1909h2_d83cc789557e_7af1ed941c26,88893,0.88893
7,t12d190800_d83cc789557e_7af1ed941c26,6300,0.06300
1,t10d120400_d94e65cdb899_f8ec56bc740a,1285,0.01285
0,t10d070700_c50f5591e341_c39ab67fec8e,1077,0.01077
23,ts3i050000_d30d14220708_e3b0c44298fc,870,0.00870
10,t12d210600_b973bfd88a0e_1da50ec048a3,696,0.00696
11,t12d210700_76e208dd3e22_2dae41c691ec,229,0.00229
20,t13d1911h2_9dc949149365_d811adc85aab,168,0.00168
4,t10i120300_d94e65cdb899_f8ec56bc740a,127,0.00127
18,t13d1516h2_8daaf6152771_e5627efa2ab1,118,0.00118


## Flow-Level Statistics

Compute descriptive statistics for duration, bytes, packets, and inter-arrival times using robust measures (median, IQR, p95/p99). This captures structural and volumetric patterns of malicious flows.


In [11]:
from tls_profiling.exploration.flow_stats import compute_flow_stats
compute_flow_stats(df)

,property,mean,std,median,iqr,p95,p99
0,duration_s,8.323189,21.088263,1.845003,3.850887,41.751099,1.102076e+02
1,bytes_up,5890.978760,11354.971306,2842.500000,4863.000000,14679.000000,6.954304e+04
2,bytes_down,54314.460200,266393.377609,9713.000000,11888.500000,44787.000000,1.799819e+06
3,bytes_total,60205.438960,275247.272996,14346.000000,18863.000000,49975.000000,1.860649e+06
4,packets_up,48.765850,191.201582,18.000000,8.000000,49.000000,1.310000e+03
5,packets_down,47.023200,193.257538,15.000000,9.000000,42.000000,1.312000e+03
6,byte_ratio_up_down,0.494326,1.071754,0.339646,0.328286,1.756957,3.273385e+00
7,iat_s,7.260328,726.947319,0.619900,1.701600,11.763080,5.484572e+01
8,pct_short_connections_lt_1s,29.089000,NaN,NaN,NaN,NaN,NaN
9,pct_large_transfers_gt_1MB,1.773000,NaN,NaN,NaN,NaN,NaN


## Traffic Structure Indicators

Report aggregate properties such as percentage of short connections and percentage of large transfers. These indicators describe behavioral tendencies (e.g., beaconing vs bulk transfer). 

In [12]:
from tls_profiling.exploration.imbalance import get_class_imbalance

imbalance_report = get_class_imbalance(connection_per_malware, "meta.malware.family")

print("Imbalance report:")
display(imbalance_report)

# 10 application with most connections
print("Most connections:")
display(connection_per_malware.sort_values("malware_connections", ascending=False).head(10))

# 10 applications with least connections
print("Least connections:")
display(connection_per_malware.sort_values("malware_connections", ascending=True).head(10))

Imbalance report:


,metric,value
0,num_classes,27.000000
1,total_connections,585.000000
2,top10_share_pct,74.017094
3,gini_coefficient,0.501931
4,num_classes_lt_50,23.000000
5,share_connections_classes_lt_50_pct,51.111111


Most connections:


,meta.malware.family,connections,system_connections,unknown_connections,application_connections,malware_connections,processes
8,formbook,75,0,0,0,75,[]
25,xworm,75,0,0,0,75,[]
0,agenttesla,70,0,0,0,70,[]
2,asyncrat,66,0,0,0,66,[]
15,modiloader,30,0,0,0,30,[]
6,cybergate,29,0,0,0,29,[]
5,blackmoon,28,0,0,0,28,[]
11,lokibot,21,0,0,0,21,[]
7,darkcomet,20,0,0,0,20,[]
22,smokeloader,19,0,0,0,19,[]


Least connections:


,meta.malware.family,connections,system_connections,unknown_connections,application_connections,malware_connections,processes
12,lumma,1,0,0,0,1,[]
17,njrat,3,0,0,0,3,[]
26,zgrat,3,0,0,0,3,[]
4,betabot,5,0,0,0,5,[]
10,hawkeye_reborn,5,0,0,0,5,[]
23,xmrig,5,0,0,0,5,[]
13,m00nd3v_logger,5,0,0,0,5,[]
1,ardamax,6,0,0,0,6,[]
21,sality,10,0,0,0,10,[]
24,xtremerat,10,0,0,0,10,[]


## Outlier Exploration

Identify extreme flows based on percentile thresholds to examine tail behavior and detect unusual or atypical malicious connections.

In [13]:
from tls_profiling.exploration.outliers import get_outliers, get_rare_values

outliers_report = get_outliers(df)

print("Outliers report:")
display(outliers_report["summary"])
tls_cols = [ "tls.cver" ,"tls.sver", "tls.sext", "tls.csg", "tls.ccs", "tls.cext", "tls.ssv", "tls.csv", "tls.scs", "tls.alpn", "tls.sni" ]
for c in tls_cols:
    display(f"Rare values for {c}:")
    display(get_rare_values(df,c,5))


Outliers report:


,metric,value
0,duration_p99.9_threshold_s,1.451703e+02
1,bytes_total_p99.9_threshold,2.804081e+06
2,n_top_duration,1.000000e+02
3,n_largest_transfer,1.000000e+02


'Rare values for tls.cver:'

,tls.cver,count,ratio
0,0302,19,0.00019
1,0300,870,0.00870
2,0301,2584,0.02584
3,0303,96527,0.96527


'Rare values for tls.sver:'

...

,tls.scs,count,ratio
0,003C,1,0.00001
1,0035,2,0.00002
2,CCA8,2,0.00002
3,1302,40,0.00040
4,C009,107,0.00107


'Rare values for tls.alpn:'

,tls.alpn,count,ratio
0,,10808,0.10808
1,"h2,http/1.1",89192,0.89192


'Rare values for tls.sni:'

,tls.sni,count,ratio
0,account.shodan.io,1,0.00001
1,api.auth.gg,1,0.00001
2,analytics.avcdn.net,1,0.00001
3,accypg.sn.files.1drv.com,1,0.00001
4,cdn.fwupd.org,1,0.00001
